# TCC WBE-IoT — Modelagem v2

Este notebook executa validação temporal expansiva, ajuste controlado de hiperparâmetros e análise de sensibilidade. **O teste de 2024 não é aberto.** Tempo estimado: 7–15 minutos, dependendo do ambiente do Colab.

## 1. Instalar dependências

In [ ]:
%pip -q install "pandas>=2.2,<3" "numpy>=1.26,<3" "pyarrow>=15,<22" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4" "joblib>=1.4,<2"

## 2. Clonar ou atualizar a branch de modelagem

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/samuelfzst-bit/tcc-vigilancia-wbe-iot.git"
GIT_REF = "feature/pipeline-modelagem"
REPO_DIR = Path("/content/tcc-vigilancia-wbe-iot")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", GIT_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", GIT_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", GIT_REF], cwd=REPO_DIR, check=True)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repositório pronto:", commit)

## 3. Montar o Drive e localizar o snapshot congelado

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = Path("/content/drive/MyDrive/TCC/datasets")
RUNS_DIR = Path("/content/drive/MyDrive/TCC/model_runs_v2")
DATASET_PATH = DATASET_DIR / "dataset_modelagem_v1_20260927_20261006T182348Z.parquet"
EXPECTED_SHA256 = "1d51c54f1d13e28b9253b6db4fa12d9538ed774da6c5d5c33b52f2cd026fcb85"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
assert DATASET_PATH.exists(), f"Snapshot não encontrado: {DATASET_PATH}"
print("Snapshot encontrado:", DATASET_PATH)

## 4. Validar a identidade do arquivo

In [ ]:
import hashlib

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

actual_sha256 = file_sha256(DATASET_PATH)
assert actual_sha256 == EXPECTED_SHA256, "O Parquet não corresponde ao snapshot aprovado."
print("SHA-256 aprovado:", actual_sha256)

## 5. Executar a modelagem v2

A célula pode levar alguns minutos e não mostra progresso contínuo. Aguarde a mensagem `Execução v2 salva`.

In [ ]:
import os
import sys

before = {p for p in RUNS_DIR.iterdir() if p.is_dir()}
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR)
subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts/train_models_v2.py"), str(DATASET_PATH), "--output", str(RUNS_DIR)],
    cwd=REPO_DIR, env=env, check=True,
)
after = {p for p in RUNS_DIR.iterdir() if p.is_dir()}
new_runs = sorted(after - before)
assert new_runs, "A execução terminou sem criar a pasta de resultados."
RUN_DIR = new_runs[-1]
print("Resultados:", RUN_DIR)

## 6. Conferir o resumo e os finalistas

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

display(Markdown((RUN_DIR / "RESUMO_EXECUCAO.md").read_text(encoding="utf-8")))
ranking = pd.read_csv(RUN_DIR / "05_ranking_finalistas.csv")
display(ranking.loc[ranking["ranking_horizonte"].le(5)])
display(Image(filename=str(RUN_DIR / "08_previsoes_campeoes_2023.png")))

## 7. Compactar os resultados para envio

In [ ]:
import shutil

zip_path = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR))
print("ZIP criado:", zip_path)
print("Envie este ZIP para a análise. Não execute nenhuma rotina de teste de 2024.")